In [ ]:
import json
import requests
from utils import get_all_related_concepts, get_api_domain, get_headers
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import deque

In [85]:
with open("all_uvl-drug-entry_references.json", "r") as f:
    data = json.load(f)

references = list(map(lambda x: x["uri"], data))

print(references)


['/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912618/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912617/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912616/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912615/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912614/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912613/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912612/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912611/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912610/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912609/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912608/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912607/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912606/', '/orgs/UVL-Burundi/collections/uvl-drug-entry/references/10912605/', '/orgs/UVL-Burundi/collections/uv

In [ ]:
def fetch_all_mappings_for_concept(concept):
    url = urljoin(urljoin(get_api_domain(), concept), "mappings/")
    params = {"limit": 100}
    mappings = []
    while url:
        response = requests.get(url, params=params, headers=get_headers())
        response.raise_for_status()
        data = response.json()
        mappings.extend(data)
        url = response.headers.get("next")
        params = {}
    mappings = list(filter(lambda x: x["map_type"] in {"CONCEPT-SET", "Q-AND-A"}, mappings))
    ciel_present = any(x['to_source_owner'] == 'CIEL' for x in mappings)
    if ciel_present:
        print(f"CIEL present in {concept}")
    return mappings

def get_concept_details(concept):
    url = urljoin(get_api_domain(), concept)
    response = requests.get(url, headers=get_headers())
    response.raise_for_status()
    data = response.json()
    return data


def get_all_related_concepts(concepts, executor=None):
    concepts = deque(concepts)
    print(len(concepts))
    all_concepts = {}
    cache = set()
    while len(concepts) > 0:
        print(len(concepts))
        concept = concepts.popleft()
        concept_details = get_concept_details(concept)
        all_concepts[concept] = concept_details
        cache.add(concept)
        mappings = fetch_all_mappings_for_concept(concept)
        mapped_concepts = list(
            map(lambda mapping: mapping["to_concept_url"], mappings)
        )
        mapped_concepts.extend(list(
            map(lambda mapping: mapping["from_concept_url"], mappings)
        ))
        new_concepts = list(filter(lambda x: x not in cache, mapped_concepts))
        concepts.extend(new_concepts)
        # all_concepts.update(new_concepts)
    return {
        "concepts": list(all_concepts.keys()),
        "concept_details": all_concepts
    }

# def get_all_related_concepts_wrapper(concepts):
#     with ThreadPoolExecutor(max_workers=20) as executor:
#         return get_all_related_concepts(concepts, executor)
    
    # futures = [
    #     executor.submit(fetch_all_mappings_for_concept, concept)
    #     for concept in concepts
    # ]
    # for future in as_completed(futures):
    #     mappings = future.result()
    #     mapped_concepts = list(
    #         map(lambda mapping: mapping["to_concept_url"], mappings)
    #     )
    #     mapped_concepts.extend(list(
    #         map(lambda mapping: mapping["from_concept_url"], mappings)
    #     ))
    #     all_concepts.update(get_all_related_concepts(mapped_concepts, executor))
    # fetched_cache.clear()
    # return list(all_concepts)




In [ ]:
concepts = [
"/orgs/UVL-Burundi/sources/uvl-custom/concepts/7898090/"
]

clear_cache()
all_related_concepts = get_all_related_concepts(concepts)
print(all_related_concepts)
print(len(all_related_concepts))

1
1
61
62
61
60
63
62
61
60
59
58
57
56
58
60
67
66
65
64
63
66
65
66
73
72
71
79
78
82
81
80
79
78
77
79
78
77
76
75
74
75
74
73
72
71
72
71
70
72
74
76
78
80
79
78
77
76
79
78
77
76
81
91
93
92
91
93
96
95
94
93
92
91
90
89
88
87
86
85
84
83
82
81
80
79
78
77
76
75
74
73
72
71
70
69
68
67
66
65
64
63
62
61
60
59
58
57
56
55
54
53
52
51
50
49
48
47
46
45
44
43
42
41
40
39
38
37
36
35
34
33
32
31
30
29
28
27
26
25
24
23
22
21
20
19
18
17
16
15
14
13
12
11
10
9
8
7
6
5
4
3
2
1
{'concepts': ['/orgs/UVL-Burundi/sources/uvl-custom/concepts/7898090/', '/orgs/UVL-Burundi/sources/uvl-custom/concepts/7898092/', '/orgs/UVL-Burundi/sources/uvl-custom/concepts/1001/', '/orgs/UVL-Burundi/sources/uvl-ciel/concepts/7896239/', '/orgs/UVL-Burundi/sources/uvl-custom/concepts/1003/', '/orgs/UVL-Burundi/sources/uvl-custom/concepts/1009/', '/orgs/UVL-Burundi/sources/uvl-ciel/concepts/7896237/', '/orgs/UVL-Burundi/sources/uvl-custom/concepts/1005/', '/orgs/UVL-Burundi/sources/uvl-ciel/concepts/7896247/', '

In [ ]:
concept_details = all_related_concepts['concept_details']
print(set(list(map(lambda x: x['concept_class'], concept_details.values()))))

{'Misc', 'Test', 'Diagnosis', 'Finding', 'LabSet', 'Question', 'ConvSet'}


In [101]:
test_concepts = list(filter(lambda x: x[1]["concept_class"] in {"Test", 'LabSet'}, concept_details.items()))
print(len(test_concepts))
test_concepts_map = {k: v for k, v in test_concepts}
print(json.dumps(list(map(lambda x: x['display_name'], test_concepts_map.values()))))
print(list(test_concepts_map.keys()))



84
["BI03 - C-Reactive Protein", "BP08 - Examination of other secretions", "BI11 - Total proteins", "HS10 - VDRL-TPHA", "BI17 - Amylasemia", "BI02 - Bilirubin (conjugated + total)", "BI22 - Peritoneal fluid analysis", "HS24 - Erythrocyte sedimentation rate (ESR)", "BI20 - Glycated albumin", "BI07 - Hemoglobin", "HS14 - DNA analysis", "HS25 - Hepatitis A antigen", "BP01 - Thick smear", "HS04 - Rapid malaria test", "BI18 - Amylasuria", "BI13 - Urea", "BI15 - Blood albumin", "BI05 - Blood glucose", "HS07 - Pregnancy test", "BP07 - Urine examination", "HS09 - Widal test", "HS12 - Blood type", "BI12 - GPT (ALT)", "BI19 - Lactic acid", "BP04 - Tuberculosis bacilli", "BI12 - GOT (AST)", "BP06 - Stool examination", "BI10 - Alkaline phosphatase", "BI14 - Creatinine", "BI16 - Serum albumin", "HS11 - CD4 count", "BP05 - CSF examination", "HS17 - Hepatitis B surface antigen", "BI06 - Glycated hemoglobin", "BI09 - CSF (albumin, glucose)", "BI01 - Uric acid", "BI21 - Amniotic fluid analysis", "HS21 

In [111]:
def fetch_all_versions(source_or_collection_url):
    url = urljoin(urljoin(get_api_domain(), source_or_collection_url), "versions/")
    response = requests.get(url, headers=get_headers())
    response.raise_for_status()
    data = response.json()
    return data

urls = [
  "/orgs/UVL-Burundi/sources/uvl-custom/",
  "/orgs/UVL-Burundi/sources/uvl-ciel/",
]

versions = []

for url in urls:
    data = fetch_all_versions(url)
    versions.extend(data)

print(len(versions))

filtered_versions = list(map(lambda x: x['version_url'], list(filter(lambda x: x['version'] != 'HEAD', versions))))

print(filtered_versions)
print(len(filtered_versions))

18
['/orgs/UVL-Burundi/sources/uvl-custom/v12/', '/orgs/UVL-Burundi/sources/uvl-custom/v11/', '/orgs/UVL-Burundi/sources/uvl-custom/v9/', '/orgs/UVL-Burundi/sources/uvl-ciel/v13/', '/orgs/UVL-Burundi/sources/uvl-ciel/v12/', '/orgs/UVL-Burundi/sources/uvl-ciel/v11/', '/orgs/UVL-Burundi/sources/uvl-ciel/v10/', '/orgs/UVL-Burundi/sources/uvl-ciel/v9/', '/orgs/UVL-Burundi/sources/uvl-ciel/v8/', '/orgs/UVL-Burundi/sources/uvl-ciel/v7/', '/orgs/UVL-Burundi/sources/uvl-ciel/v6/', '/orgs/UVL-Burundi/sources/uvl-ciel/v5/', '/orgs/UVL-Burundi/sources/uvl-ciel/v4/', '/orgs/UVL-Burundi/sources/uvl-ciel/v3/', '/orgs/UVL-Burundi/sources/uvl-ciel/v2/', '/orgs/UVL-Burundi/sources/uvl-ciel/v1/']
16
